# Notebook Demo: Web Content Mining: Crawling and Scraping  

This notebook demonstrates a **basic web content mining pipeline using Python**.

Students will learn how to:

- Send HTTP requests to download web pages  
- Parse HTML content using Python libraries  
- Extract information such as text, authors, and links  
- Follow hyperlinks to collect data from multiple pages  
- Organize extracted data into structured datasets  
- Export data for further data mining analysis  

Goal: understand how **web crawling and scraping support real-world data mining workflows**.

---

## 1. Install or import required libraries

In most teaching environments, the following packages are commonly used:

- `requests` for downloading web pages
- `beautifulsoup4` for parsing HTML
- `pandas` for tabular data handling
- `re` for regular expressions
- `urllib.parse` for joining and normalizing URLs

The install command is included as a comment in case students need it.

In [ ]:
# If needed, uncomment the next line:
# !pip install requests beautifulsoup4 pandas lxml

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from urllib.parse import urljoin, urlparse
from collections import Counter, deque

print("Libraries imported successfully.")

## 2. Define target URLs for the classroom demo

For classroom demonstrations, it is better to use stable and scraping-friendly example sites.

We will use:
- `quotes.toscrape.com` for quotes, authors, and tags
- `books.toscrape.com` for books, categories, and prices

These are widely used in instructional scraping examples.

In [ ]:
QUOTES_BASE = "https://quotes.toscrape.com/"
BOOKS_BASE = "https://books.toscrape.com/"

print("Quotes demo site:", QUOTES_BASE)
print("Books demo site:", BOOKS_BASE)

## 3. Download a web page with `requests`

The first step in scraping is usually to send an HTTP GET request and check whether the server responds successfully.

Important response attributes:
- `status_code`: HTTP status (200 means success)
- `headers`: metadata about the response
- `text`: HTML content of the page

In [ ]:
response = requests.get(QUOTES_BASE, timeout=20)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("\nFirst 500 characters of the page:\n")
print(response.text[:500])

## 4. Parse HTML with BeautifulSoup

`BeautifulSoup` converts raw HTML into a parse tree that we can search using:
- tag names
- CSS classes
- attributes
- nested structures

This is the main tool used for many beginner and intermediate scraping tasks.

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

print("Page title tag:")
print(soup.title)

print("\nPage title text:")
print(soup.title.get_text(strip=True))

## 5. Inspect the structure of the page

Before extracting data, it is good practice to inspect repeated page elements.

On the quotes site, each quote is contained inside a repeated `div` element with class `quote`.

In [ ]:
quote_blocks = soup.find_all("div", class_="quote")

print("Number of quote blocks found on page 1:", len(quote_blocks))
print("\nPreview of the first quote block:")
print(quote_blocks[0].prettify()[:1200])

## 6. Extract quote text from repeated blocks

A common scraping pattern is:
1. find repeated containers
2. inspect internal tags
3. extract target fields

Here we collect the quote text from each quote block.

In [ ]:
quote_texts = []

for block in quote_blocks:
    text_tag = block.find("span", class_="text")
    if text_tag:
        quote_texts.append(text_tag.get_text(strip=True))

print("Extracted quote texts:")
for i, qt in enumerate(quote_texts[:5], start=1):
    print(f"{i}. {qt}")

## 7. Extract multiple fields: quote, author, and tags

Real scraping tasks usually require more than one field.
We now extract:
- quote text
- author name
- tags associated with the quote

In [ ]:
records = []

for block in quote_blocks:
    text_tag = block.find("span", class_="text")
    author_tag = block.find("small", class_="author")
    tag_tags = block.find_all("a", class_="tag")

    record = {
        "quote": text_tag.get_text(strip=True) if text_tag else None,
        "author": author_tag.get_text(strip=True) if author_tag else None,
        "tags": [t.get_text(strip=True) for t in tag_tags]
    }
    records.append(record)

records[:3]

## 8. Convert scraped records into a DataFrame

After extraction, we usually convert data into a tabular form for:
- cleaning
- filtering
- analysis
- export

In [ ]:
df_quotes = pd.DataFrame(records)
df_quotes.head()

## 9. Basic data inspection

Once content is structured, we can perform basic exploratory checks:
- number of rows
- missing values
- distinct authors
- average tag count

In [ ]:
print("Shape:", df_quotes.shape)
print("\nMissing values:")
print(df_quotes.isna().sum())

print("\nUnique authors on page 1:", df_quotes["author"].nunique())
print("Average number of tags per quote:", df_quotes["tags"].apply(len).mean())

## 10. Extract all hyperlinks from a page

Crawling depends on link extraction.
A crawler typically:
- downloads a page
- extracts links
- normalizes them into absolute URLs
- adds them to a queue or frontier

In [ ]:
all_links = soup.find_all("a", href=True)
normalized_links = []

for a in all_links:
    absolute_url = urljoin(QUOTES_BASE, a["href"])
    normalized_links.append(absolute_url)

print("Total links found:", len(normalized_links))
print("\nSample links:")
for link in normalized_links[:10]:
    print(link)

## 11. Filter internal links only

In many crawling tasks, we do not want to follow every discovered link.
Instead, we often restrict the crawl to:
- one domain
- one subsite
- one topic

This is called a **crawl boundary**.

In [ ]:
def is_internal(url, base_domain):
    return urlparse(url).netloc == base_domain

base_domain = urlparse(QUOTES_BASE).netloc
internal_links = sorted(set([u for u in normalized_links if is_internal(u, base_domain)]))

print("Internal links found:", len(internal_links))
for link in internal_links[:15]:
    print(link)

## 12. Handle pagination manually

Many websites split content across multiple pages.
A scraper may need to:
- identify the “next” button
- follow the next page link
- repeat until no next page exists

In [ ]:
next_button = soup.find("li", class_="next")
next_url = None

if next_button and next_button.find("a"):
    next_url = urljoin(QUOTES_BASE, next_button.find("a")["href"])

print("Next page URL:", next_url)

## 13. Write a function to scrape one quotes page

Reusable functions are essential for scalable mining workflows.
This function downloads one page and returns structured quote records.

In [ ]:
def scrape_quotes_page(url):
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    page_records = []
    for block in soup.find_all("div", class_="quote"):
        text_tag = block.find("span", class_="text")
        author_tag = block.find("small", class_="author")
        tag_tags = block.find_all("a", class_="tag")

        page_records.append({
            "source_url": url,
            "quote": text_tag.get_text(strip=True) if text_tag else None,
            "author": author_tag.get_text(strip=True) if author_tag else None,
            "tags": [t.get_text(strip=True) for t in tag_tags]
        })

    return page_records, soup

page1_records, page1_soup = scrape_quotes_page(QUOTES_BASE)
print("Records from page 1:", len(page1_records))
page1_records[:2]

## 14. Crawl multiple pages through pagination

We now combine scraping and crawling:
- scrape one page
- locate the next page
- repeat until the crawl stops

This is a simple but common pattern for small-scale web content mining.

In [ ]:
all_quote_records = []
current_url = QUOTES_BASE
visited_pages = []

while current_url:
    print("Visiting:", current_url)
    page_records, page_soup = scrape_quotes_page(current_url)
    all_quote_records.extend(page_records)
    visited_pages.append(current_url)

    next_button = page_soup.find("li", class_="next")
    if next_button and next_button.find("a"):
        current_url = urljoin(current_url, next_button.find("a")["href"])
        time.sleep(1)  # polite delay
    else:
        current_url = None

print("\nTotal pages visited:", len(visited_pages))
print("Total quote records collected:", len(all_quote_records))

## 15. Build a full quotes dataset

Now we convert the full crawl output into a DataFrame and inspect the result.

In [ ]:
df_all_quotes = pd.DataFrame(all_quote_records)
df_all_quotes.head()

## 16. Expand list-valued tags into a simpler text column

Some fields are naturally lists, but for export or quick analysis we may want:
- a comma-separated string column
- or one row per tag

We first create a simple comma-separated version.

In [ ]:
df_all_quotes["tags_str"] = df_all_quotes["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")
df_all_quotes[["author", "quote", "tags_str"]].head()

## 17. Basic frequency analysis on authors and tags

Web content mining often leads directly to descriptive analysis.

Here we compute:
- most frequent authors
- most common tags

In [ ]:
author_counts = df_all_quotes["author"].value_counts()

tag_counter = Counter()
for tag_list in df_all_quotes["tags"]:
    for tag in tag_list:
        tag_counter[tag] += 1

print("Top 10 authors:")
print(author_counts.head(10))

print("\nTop 10 tags:")
print(tag_counter.most_common(10))

## 18. Scrape a different site: books.toscrape.com

A useful class demo shows that the same ideas generalize across sites.

On the books site, each book page preview contains fields such as:
- title
- price
- availability
- rating

In [ ]:
books_response = requests.get(BOOKS_BASE, timeout=20)
print("Status code:", books_response.status_code)

books_soup = BeautifulSoup(books_response.text, "html.parser")
book_articles = books_soup.find_all("article", class_="product_pod")

print("Books found on front page:", len(book_articles))
print("\nPreview of first product block:")
print(book_articles[0].prettify()[:1200])

## 19. Extract book metadata from repeated product blocks

This example reinforces the repeated-container extraction pattern.

In [ ]:
book_records = []

for article in book_articles:
    title_tag = article.find("h3").find("a")
    price_tag = article.find("p", class_="price_color")
    avail_tag = article.find("p", class_="instock availability")
    rating_tag = article.find("p", class_=re.compile("star-rating"))

    classes = rating_tag.get("class", []) if rating_tag else []
    rating = [c for c in classes if c != "star-rating"]
    rating = rating[0] if rating else None

    book_records.append({
        "title": title_tag.get("title") if title_tag else None,
        "book_url": urljoin(BOOKS_BASE, title_tag.get("href")) if title_tag else None,
        "price": price_tag.get_text(strip=True) if price_tag else None,
        "availability": " ".join(avail_tag.get_text(" ", strip=True).split()) if avail_tag else None,
        "rating": rating
    })

df_books = pd.DataFrame(book_records)
df_books.head()

## 20. Clean extracted fields

Scraped values are often strings that need conversion before analysis.

Examples:
- converting price from text to numeric
- standardizing availability text
- mapping rating words to integers

In [ ]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

# Clean price column and convert to float
df_books["price_gbp"] = (
    df_books["price"]
    .str.replace(r"[^\d.]", "", regex=True)   # remove £, Â, and other non-numeric chars
    .astype(float)
)

# Convert rating text to numeric
df_books["rating_num"] = df_books["rating"].map(rating_map)

# Preview cleaned dataset
df_books[["title", "price", "price_gbp", "rating", "rating_num"]].head()

## 21. Follow product links to scrape more detailed pages

A crawler often collects URLs first, then a scraper visits detail pages for richer content.

Here we define a function to scrape one book detail page.

In [ ]:
def scrape_book_detail(url):
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    title = soup.find("div", class_="product_main").find("h1").get_text(strip=True)
    description = None

    desc_header = soup.find("div", id="product_description")
    if desc_header:
        next_p = desc_header.find_next("p")
        if next_p:
            description = next_p.get_text(" ", strip=True)

    table = soup.find("table", class_="table table-striped")
    info = {}
    if table:
        for row in table.find_all("tr"):
            th = row.find("th")
            td = row.find("td")
            if th and td:
                info[th.get_text(strip=True)] = td.get_text(strip=True)

    return {
        "title": title,
        "description": description,
        **info
    }

sample_book_url = df_books.loc[0, "book_url"]
sample_book_detail = scrape_book_detail(sample_book_url)
sample_book_detail

## 22. Create a small detailed dataset from multiple book pages

For a classroom demo, we usually do not need to crawl the full site.
Instead, we can collect a small sample of detail pages and build a richer dataset.

In [ ]:
detailed_books = []

for url in df_books["book_url"].head(5):
    print("Scraping detail page:", url)
    detailed_books.append(scrape_book_detail(url))
    time.sleep(1)

df_book_details = pd.DataFrame(detailed_books)
df_book_details.head()

## 23. Save scraped datasets to CSV and JSON

Once the data has been collected and cleaned, it can be exported for:
- later analysis
- dashboards
- machine learning pipelines
- classroom assignments

In [ ]:
df_all_quotes_export = df_all_quotes.copy()
df_all_quotes_export["tags"] = df_all_quotes_export["tags"].apply(lambda x: ", ".join(x))

df_all_quotes_export.to_csv("quotes_dataset.csv", index=False)
df_books.to_csv("books_front_page.csv", index=False)
df_book_details.to_json("book_details.json", orient="records", indent=2)

print("Files saved:")
print("- quotes_dataset.csv")
print("- books_front_page.csv")
print("- book_details.json")

## 24. Build a very small focused crawler

A crawler usually maintains:
- a queue (frontier) of URLs to visit
- a visited set
- rules that decide which links are allowed

The function below demonstrates a small breadth-first crawl restricted to one domain.

In [ ]:
def simple_crawl(start_url, max_pages=5, delay=1):
    domain = urlparse(start_url).netloc
    queue = deque([start_url])
    visited = set()
    crawl_log = []

    while queue and len(visited) < max_pages:
        url = queue.popleft()
        if url in visited:
            continue

        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            visited.add(url)

            title = soup.title.get_text(strip=True) if soup.title else None
            crawl_log.append({"url": url, "title": title})
            print(f"Visited ({len(visited)}/{max_pages}):", url)

            for a in soup.find_all("a", href=True):
                next_url = urljoin(url, a["href"])
                parsed = urlparse(next_url)

                if parsed.netloc == domain and next_url not in visited:
                    queue.append(next_url)

            time.sleep(delay)

        except Exception as e:
            print("Error:", url, "|", e)

    return pd.DataFrame(crawl_log)

df_crawl_demo = simple_crawl(QUOTES_BASE, max_pages=5, delay=1)
df_crawl_demo

## Summary & Discussion

This notebook demonstrated a basic **web content mining workflow**:
- Sending HTTP requests  
- Parsing HTML  
- Extracting repeated content  
- Handling pagination and links  
- Building and cleaning datasets  
- Exporting data for analysis  

**Key Takeaways**
- **Crawling** discovers and visits web pages  
- **Scraping** extracts useful information  
- Web mining converts **raw web pages into structured datasets**

### Discussion Questions and Brief Answers

**1. What makes a website easy or difficult to scrape?**  
Websites with simple HTML structure are easier, while dynamic content, JavaScript rendering, logins, or anti-scraping protections make scraping harder.

**2. Why is pagination important in web mining?**  
Pagination allows collecting **large datasets distributed across multiple pages**.

**3. What are the risks of scraping at large scale?**  
Possible server overload, IP blocking, legal issues, or violating website policies.

**4. How can scraped text be used in data mining?**  
It can support **clustering, topic modeling, sentiment analysis, and information extraction**.

# See you next lecture!